# 실험 E1 (Colab) — Qwen3.5-9B 제출 추론

로컬 노트북 [AI_Challenge2_expE1_qwen35_9b.ipynb](AI_Challenge2_expE1_qwen35_9b.ipynb)의
**제출 추론 섹션**을 Colab용으로 옮긴 것입니다.
프롬프트·이미지 규칙·파서는 `src/common_eval.py`를 그대로 import해서 씁니다.

## 어떤 GPU를 고를까

| 런타임 | VRAM | bf16 | 9B 적재 방식 | 평가 |
| --- | --- | --- | --- | --- |
| **A100 GPU** | 40 GB | ○ | **bf16 그대로 (18.0 GiB)** | **권장.** 양자화 오버헤드 없이 가장 빠름 |
| L4 GPU | 24 GB | ○ | bf16은 여유 3 GiB뿐 → 기본 4bit | 돌아가지만 A100보다 느림 |
| T4 GPU | 16 GB | ✗ | 4bit 필수, 계산은 fp16 | 가장 느림. Turing이라 bf16 미지원 |
| v6e-1 / v5e-1 TPU | — | — | — | bitsandbytes·Qwen3.5 VLM 경로가 TPU를 지원하지 않음. 쓰지 마세요 |

**A100 + bf16을 권장합니다.** 이유는 세 가지입니다.

1. 로컬에서 4bit를 쓴 건 16 GiB GPU에 bf16 18.0 GiB가 안 올라가서지, 4bit가 빨라서가 아닙니다.
   bitsandbytes NF4는 행렬곱마다 역양자화가 붙어 **VRAM이 충분하면 bf16이 더 빠릅니다**
2. A100 40 GB는 bf16 가중치 18 GiB에 활성값까지 넉넉합니다
3. Qwen3.5는 linear-attention 계열 계층을 씁니다. Colab(Linux)에서는
   `flash-linear-attention`을 설치할 수 있어, 로컬 Windows에서 "reference PyTorch implementation"으로
   떨어지던 `chunk_gated_delta_rule`이 최적화 커널로 돕니다

아래 설정 셀이 **런타임을 감지해 4bit 여부를 자동으로 정하므로**, L4나 T4를 골라도 그대로 실행됩니다.

## 실행 순서

1. `런타임 - 런타임 유형 변경`에서 **A100 GPU** 선택
2. 1번 설치 셀 실행 → **세션 다시 시작** → 2번부터 이어서 실행
3. 속도 측정 셀에서 예상 시간을 확인한 뒤 본 추론 실행

## 1. 패키지 설치

`transformers 5.x`가 필요합니다(Colab 기본은 4.x). `flash-linear-attention`은 Triton 기반이라
컴파일 없이 빠르게 깔립니다.

`causal-conv1d`는 소스 빌드라 10분 이상 걸리거나 실패할 수 있어 기본으로 넣지 않았습니다.
이쪽은 짧은 conv 경로라 영향이 작습니다.

**설치 후 반드시 `런타임 - 세션 다시 시작`을 하고, 이 셀은 건너뛴 채 2번부터 실행하세요.**

In [ ]:
!pip -q install "transformers>=5.3.0,<6" accelerate bitsandbytes
!pip -q install flash-linear-attention
!pip -q install pillow pandas tqdm hf_transfer

# 소스 빌드라 오래 걸립니다. 필요하면 주석을 푸세요.
# !pip -q install causal-conv1d

In [ ]:
# 이후 모든 셀이 함께 쓰는 import를 여기서 한 번에 합니다.
import gc, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

import pandas as pd
import torch, transformers
from tqdm.auto import tqdm

print("transformers", transformers.__version__, "/ torch", torch.__version__)
assert transformers.__version__ >= "5", "세션을 다시 시작한 뒤 2번 셀부터 실행하세요."
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요 (런타임 - 런타임 유형 변경)."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_OK = torch.cuda.is_bf16_supported()
print(f"GPU: {GPU_NAME} / {GPU_GIB:.1f} GiB / bf16 {BF16_OK}")

try:
    import fla                      # noqa: F401
    print("flash-linear-attention: 설치됨 (최적화 커널 사용)")
except ImportError:
    print("flash-linear-attention: 없음 -> 참조 구현으로 동작합니다 (정확하지만 느림)")

## 2. Drive 마운트 · 데이터 준비

Drive에 대회 데이터 zip이 있어야 합니다. 경로는 OCR Colab 노트북과 같은 규칙입니다.
결과도 Drive에 남겨 런타임이 끊겨도 이어서 돌릴 수 있게 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")

assert os.path.isfile(ZIP_PATH), f"zip을 찾지 못했습니다: {ZIP_PATH}"


def find_data_root():
    """test.csv가 들어 있는 폴더를 데이터 루트로 봅니다."""
    for parent, dirs, files in os.walk('/content'):
        dirs[:] = [d for d in dirs if d not in {'drive', '.cache', 'sample_data', 'models'}]
        if 'test.csv' in files and 'train.csv' in files:
            return Path(parent)
    return None


if find_data_root() is None:
    !7z x -y -p"$ZIP_PASSWORD" -o/content/ "$ZIP_PATH"

DATA_ROOT = find_data_root()
assert DATA_ROOT is not None, "압축 해제 후에도 train.csv/test.csv를 찾지 못했습니다."
print("데이터 루트:", DATA_ROOT)
print("파일:", sorted(p.name for p in DATA_ROOT.iterdir())[:10])

## 3. 공통 코드 가져오기

프롬프트·파서·이미지 규칙이 로컬 실험과 **글자 단위로 같아야** 결과를 비교할 수 있습니다.
그래서 노트북에 복사하지 않고 `src/common_eval.py`를 그대로 import합니다.

**저장소가 비공개라 Colab에서 `git clone`이 그냥은 안 됩니다**
(`could not read Username for 'https://github.com'`). 아래 순서로 찾습니다.

1. **Drive 사본** — `MyDrive/SSAFY/AI_Challenge2/src/common_eval.py`.
   토큰이 필요 없어 가장 간단합니다. **이 방법을 권장합니다**
2. **clone** — 저장소가 공개이거나, Colab 비밀에 `GITHUB_TOKEN`을 넣어 둔 경우
   (왼쪽 열쇠 아이콘 🔑 → 새 비밀 → 이름 `GITHUB_TOKEN`, 값은 `repo` 권한 PAT → 노트북 액세스 켜기)
3. **직접 업로드** — 위 둘 다 없으면 파일 선택 창이 뜹니다

1번을 쓰려면 로컬에서 `src/common_eval.py` 한 파일만 Drive의 위 경로에 올려 두세요.

In [ ]:
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Hun2e/AI_Challenge.git"
REPO_DIR = Path("/content/AI_Challenge")
DRIVE_SRC = DRIVE_BASE / "src" / "common_eval.py"   # 1번 경로
SRC_DIR = None


def _ok(d):
    return d is not None and (Path(d) / "common_eval.py").is_file()


# 0) 이미 받아 둔 게 있으면 재사용 (셀을 다시 실행할 때)
for cand in (Path("/content/src"), REPO_DIR / "src"):
    if _ok(cand):
        SRC_DIR = cand
        print("이미 준비됨:", SRC_DIR)
        break

# 1) Drive 사본 — 비공개 저장소여도 토큰이 필요 없습니다
if SRC_DIR is None and DRIVE_SRC.is_file():
    SRC_DIR = Path("/content/src")
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SRC, SRC_DIR / "common_eval.py")
    print("Drive 사본 사용:", DRIVE_SRC)

# 2) clone — 공개 저장소이거나 Colab 비밀에 GITHUB_TOKEN이 있을 때
if SRC_DIR is None:
    url, token = REPO_URL, None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    done = subprocess.run(["git", "clone", "--depth", "1", "-q", url, str(REPO_DIR)],
                          capture_output=True, text=True)
    if done.returncode != 0:
        # git 오류 메시지에 URL이 섞여 나올 수 있으므로 토큰을 가립니다.
        msg = (done.stderr or "").strip()
        if token:
            msg = msg.replace(token, "***")
        print("clone 실패:", msg.splitlines()[0] if msg else f"returncode {done.returncode}")
    if _ok(REPO_DIR / "src"):
        SRC_DIR = REPO_DIR / "src"
        print("clone 사용:", SRC_DIR)

# 3) 직접 업로드 (Colab에서만 동작)
if SRC_DIR is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is not None:
        print("common_eval.py를 직접 올려주세요 (로컬 저장소의 src/common_eval.py)")
        uploaded = files.upload()
        assert "common_eval.py" in uploaded, "common_eval.py가 아닙니다"
        SRC_DIR = Path("/content/src")
        SRC_DIR.mkdir(parents=True, exist_ok=True)
        (SRC_DIR / "common_eval.py").write_bytes(uploaded["common_eval.py"])

assert _ok(SRC_DIR), (
    "common_eval.py를 찾지 못했습니다. "
    f"로컬의 src/common_eval.py를 {DRIVE_SRC} 로 복사한 뒤 이 셀을 다시 실행하세요.")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import common_eval as ce

# ce.DATA_DIR은 기본이 상대경로 "data"입니다. Colab의 실제 위치로 바꿔 줍니다.
ce.DATA_DIR = DATA_ROOT
ce._self_test()
print("common_eval OK / DATA_DIR =", ce.DATA_DIR)
print("이미지 긴 변 제한:", ce.IMAGE_MAX_EDGE, "px (로컬 실험과 동일)")


## 4. 런타임에 맞춘 설정

VRAM을 보고 4bit 여부를 정합니다.

- **30 GiB 이상 + bf16 지원** (A100, H100) → bf16 그대로. 가장 빠름
- 그 외 (L4, T4) → 4bit NF4. bf16 18.0 GiB가 안 들어가거나 여유가 없음

L4에서 bf16을 굳이 시도하려면 `FORCE_BF16 = True`로 두세요. 여유가 3 GiB뿐이라 OOM 위험이 있습니다.

In [ ]:
FORCE_BF16 = False    # L4 등에서 bf16을 강제할 때만 True

VLM_4BIT = not (BF16_OK and (GPU_GIB >= 30 or FORCE_BF16))
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
MAX_NEW_TOKENS = 24        # 로컬 실험과 동일
ENABLE_THINKING = False    # 켜면 출력이 길어져 parse_choice가 전부 실패합니다

torch.manual_seed(42)
print(f"{GPU_NAME}: 4bit={VLM_4BIT} / dtype={DTYPE}")
if VLM_4BIT:
    print("  -> 양자화 경로입니다. A100을 쓰면 bf16으로 더 빨라집니다")
else:
    print("  -> bf16 그대로 올립니다 (가중치 약 18.0 GiB)")

## 5. 모델 다운로드

`Qwen/Qwen3.5-9B`를 `/content/models`에 받습니다. 약 18 GB라 몇 분 걸립니다.
`hf_transfer`로 병렬 다운로드를 켭니다. 런타임을 새로 잡을 때마다 다시 받아야 합니다.

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download

VLM_MODEL_DIR = Path("/content/models/Qwen3.5-9B")
if not (VLM_MODEL_DIR / "config.json").is_file():
    snapshot_download(repo_id="Qwen/Qwen3.5-9B", local_dir=str(VLM_MODEL_DIR))

index = VLM_MODEL_DIR / "model.safetensors.index.json"
need = sorted(set(json.loads(index.read_text())["weight_map"].values())) if index.is_file() else []
miss = [f for f in need if not (VLM_MODEL_DIR / f).is_file()]
assert not miss, f"가중치 shard 누락: {miss[:2]}"
print("모델 준비 완료:", VLM_MODEL_DIR, f"({len(need)} shards)")

## 6. 프롬프트 · 모델 로드

로컬 노트북 셀 5·7과 동일합니다.

In [ ]:
import gc, json, time
import pandas as pd
from tqdm.auto import tqdm


def build_messages(row, image):
    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},
            {"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": ce.make_prompt(row)}]}]


print(build_messages({"question": "(예시) 이 간판의 상호명은 무엇인가요?",
                      "a": "가게1", "b": "가게2", "c": "가게3", "d": "가게4"},
                     None)[1]["content"][1]["text"])

In [ ]:
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration


def load_vlm():
    proc = AutoProcessor.from_pretrained(str(VLM_MODEL_DIR), local_files_only=True)
    kwargs = dict(local_files_only=True, dtype=DTYPE, device_map={"": 0})
    if VLM_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(VLM_MODEL_DIR), **kwargs)
    return model.eval(), proc


@torch.inference_mode()
def generate_text(model, proc, inputs, limit):
    inputs = inputs.to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    output = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True)
    new = output[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated), int(len(new))

## 7. 제출 설정

`SUBMISSION_CSV` 한 줄만 바꾸면 대상이 바뀌고 출력 폴더도 따라갑니다.
결과는 `/content`에 쓰고 일정 간격마다 Drive로 복사하므로, 런타임이 끊겨도 이어서 돌릴 수 있습니다.

In [ ]:
# --- 제출 설정 (여기만 바꾸면 됩니다) --------------------------------------
SUBMISSION_CSV     = DATA_ROOT / "test.csv"   # 대상. train.csv를 주면 채점 모드
SUBMISSION_NAME    = None        # 이름을 직접 정할 때. None이면 아래 두 값으로 자동 생성
SUBMISSION_METHOD  = "zeroshot"  # 방식 태그
SUBMISSION_VERSION = None        # None이면 같은 모델·방식의 다음 번호
SUBMISSION_LIMIT   = None        # int면 앞 N개만 추론(배관 점검용)
RESUME             = True        # 남은 문항부터 이어서
SYNC_EVERY         = 200         # 이 건수마다 Drive로 복사
BENCH_N            = 16          # 속도 측정에 쓸 건수
# ---------------------------------------------------------------------------

STEM = SUBMISSION_CSV.stem                       # test / train
OUT_DIR = Path("/content/out") / f"exp_E1_{STEM}"
DRIVE_DIR = DRIVE_BASE / "output" / f"exp_E1_{STEM}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = OUT_DIR / "predictions.jsonl"
DRIVE_PRED = DRIVE_DIR / "predictions.jsonl"
META_PATH = OUT_DIR / "run_meta.json"

RUN_META = {"source_csv": SUBMISSION_CSV.name, "model": VLM_MODEL_DIR.name,
            "vlm_4bit": VLM_4BIT, "max_new_tokens": MAX_NEW_TOKENS,
            "enable_thinking": ENABLE_THINKING, "image_max_edge": ce.IMAGE_MAX_EDGE}

# 이전 세션에서 Drive에 남긴 예측이 있으면 가져와 이어서 진행합니다.
if RESUME and not PRED_PATH.is_file() and DRIVE_PRED.is_file():
    import shutil
    shutil.copy2(DRIVE_PRED, PRED_PATH)
    if (DRIVE_DIR / "run_meta.json").is_file():
        shutil.copy2(DRIVE_DIR / "run_meta.json", META_PATH)
    print("Drive에서 이전 예측을 복구했습니다")

target_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
REQUIRED = ["id", "path", "question", "a", "b", "c", "d"]
assert not (set(REQUIRED) - set(target_df.columns)), "CSV 컬럼이 부족합니다"
assert target_df["id"].is_unique, "id 중복이 있습니다"
missing_img = [p for p in target_df.path if not (ce.DATA_DIR / str(p)).is_file()]
assert not missing_img, f"이미지 누락 {len(missing_img)}개. 예: {missing_img[:3]}"

if "answer" in target_df.columns:
    GOLD = target_df.set_index("id")["answer"].astype(str).str.strip().str.lower()
    print(f"정답 컬럼 발견 -> 채점 모드 ({len(GOLD)}개)")
else:
    GOLD = None
    print("정답 컬럼 없음 -> 제출 전용 모드")

IS_TEST_SPLIT = SUBMISSION_CSV.name == "test.csv"
sample_df = (pd.read_csv(DATA_ROOT / "sample_submission.csv", keep_default_na=False)
             if IS_TEST_SPLIT else None)
if sample_df is not None:
    assert list(target_df.id) == list(sample_df.id), "sample_submission과 id 순서가 다릅니다"

target_rows = target_df.drop(columns=["answer"], errors="ignore").to_dict("records")
if SUBMISSION_LIMIT:
    target_rows = target_rows[:SUBMISSION_LIMIT]
    print(f"SUBMISSION_LIMIT={SUBMISSION_LIMIT} -> 제출 파일은 만들지 않습니다")

done_ids = set()
if RESUME and PRED_PATH.is_file():
    if META_PATH.is_file():
        prev = json.loads(META_PATH.read_text(encoding="utf-8"))
        changed = {k: f"{prev.get(k)} -> {v}" for k, v in RUN_META.items() if prev.get(k) != v}
        assert not changed, (f"기존 예측과 실행 조건이 다릅니다: {changed}. "
                             "RESUME=False로 두거나 폴더를 비우세요")
    with PRED_PATH.open(encoding="utf-8") as f:
        done_ids = {json.loads(line)["id"] for line in f if line.strip()}
    print(f"이어서 진행: 이미 끝난 {len(done_ids)}개 건너뜀")
elif not RESUME and PRED_PATH.is_file():
    PRED_PATH.unlink()

META_PATH.write_text(json.dumps(RUN_META, ensure_ascii=False, indent=2), encoding="utf-8")
todo = [r for r in target_rows if r["id"] not in done_ids]
print(f"출력: {OUT_DIR}  (Drive 사본: {DRIVE_DIR})")
print(f"추론 대상 {len(todo)}개")

## 8. 폴백 · 파일명

파싱 실패를 특정 글자로 메우면 그 글자만 과대 예측되어 분포가 무너집니다
(기존 베이스라인이 `return "a"`를 써서 `a` 42%가 됐습니다).
여기서는 a~d 로짓을 비교해 고릅니다.

In [ ]:
import re


def letter_token_ids(proc):
    """a~d를 시작하는 첫 토큰 후보. 선행 공백·대소문자 변형을 모두 모읍니다."""
    tok = proc.tokenizer
    table = {}
    for ch in ce.CHOICES:
        cand = set()
        for form in (ch, " " + ch, ch.upper(), " " + ch.upper()):
            enc = tok.encode(form, add_special_tokens=False)
            if enc:
                cand.add(int(enc[0]))
        assert cand, f"'{ch}'의 토큰 id를 찾지 못했습니다"
        table[ch] = sorted(cand)
    return table


@torch.inference_mode()
def choose_by_logit(model, proc, inputs, table):
    """생성 결과가 INVALID일 때만 호출하는 폴백."""
    inputs = inputs.to(model.device)
    logits = model(**inputs).logits[0, -1].float()
    scores = {ch: max(logits[i].item() for i in ids) for ch, ids in table.items()}
    return max(scores, key=scores.get), scores


def submission_filename():
    """submission_<모델>_<방식>_v<버전>.csv"""
    if SUBMISSION_NAME:
        name = str(SUBMISSION_NAME)
        return name if name.endswith(".csv") else name + ".csv"
    model_tag = VLM_MODEL_DIR.name.replace(".", "").lower()
    quant_tag = "4bit" if VLM_4BIT else "bf16"
    stem = f"submission_{model_tag}_{SUBMISSION_METHOD}-{quant_tag}"
    version = SUBMISSION_VERSION
    if version is None:
        pattern = re.compile(re.escape(stem) + r"_v(\d+)$")
        used = []
        for path in list(OUT_DIR.glob(stem + "_v*.csv")) + list(DRIVE_DIR.glob(stem + "_v*.csv")):
            matched = pattern.fullmatch(path.stem)
            if matched:
                used.append(int(matched.group(1)))
        version = max(used, default=0) + 1
    return f"{stem}_v{version}.csv"

## 9. 모델 적재

18 GB를 GPU에 올립니다. 1~2분 걸립니다. 이 셀은 **한 번만** 실행하세요.

In [ ]:
model, proc = load_vlm()
LETTERS = letter_token_ids(proc)
torch.cuda.reset_peak_memory_stats()
print("적재 완료 / 현재 VRAM",
      f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB / {GPU_GIB:.1f} GiB")

## 10. 추론 함수

측정 셀과 본 실행 셀이 같은 함수를 씁니다. 한 건 끝날 때마다 기록하므로
중간에 끊겨도 `RESUME`으로 이어집니다.

In [ ]:
import shutil


def run_items(items, desc):
    """items를 추론해 predictions.jsonl에 append하고, 걸린 시간 목록을 돌려줍니다."""
    times, n_fallback = [], 0
    with PRED_PATH.open("a", encoding="utf-8") as log:
        for i, row in enumerate(tqdm(items, desc=desc), 1):
            torch.cuda.synchronize(); start = time.perf_counter()
            im = ce.open_image(row)
            inputs = proc.apply_chat_template(
                build_messages(row, im), add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", enable_thinking=ENABLE_THINKING)
            raw, capped, n_tok = generate_text(model, proc, inputs, MAX_NEW_TOKENS)
            pred = ce.parse_choice(raw)
            fallback = pred == "INVALID"
            if fallback:
                pred, _ = choose_by_logit(model, proc, inputs, LETTERS)
                n_fallback += 1
            torch.cuda.synchronize(); elapsed = time.perf_counter() - start
            times.append(elapsed)
            log.write(json.dumps(dict(
                id=row["id"], raw_output=raw, pred=pred, fallback=fallback,
                answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,
                image_width=im.width, image_height=im.height), ensure_ascii=False) + "\n")
            log.flush()
            del inputs, im
            if i % SYNC_EVERY == 0:
                shutil.copy2(PRED_PATH, DRIVE_PRED)
    shutil.copy2(PRED_PATH, DRIVE_PRED)
    shutil.copy2(META_PATH, DRIVE_DIR / "run_meta.json")
    if items:
        print(f"폴백 {n_fallback}개 ({n_fallback / len(items):.2%}) / "
              f"GPU peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
    return times

## 11. 속도 측정

앞 `BENCH_N`건으로 실제 속도를 재고 전체 소요 시간을 추정합니다.
여기서 나온 예측도 그대로 저장되므로 버려지는 계산은 없습니다.

참고: 로컬 RTX 5060 Ti(4bit, 최적화 커널 없음)에서 검증 672문항이 **0.805초/건**이었습니다.

In [ ]:
import statistics

bench = run_items(todo[:BENCH_N], "속도 측정")
if bench:
    warm = bench[3:] if len(bench) > 3 else bench      # 앞 몇 건은 워밍업
    per = statistics.median(warm)
    remaining = len(todo) - len(bench)
    print(f"\n워밍업 제외 중앙값 {per:.3f}초/건 (1번째 {bench[0]:.2f}초)")
    print(f"남은 {remaining}개 예상 {per * remaining / 60:.0f}분")
    print(f"로컬 0.805초/건 대비 {0.805 / per:.1f}배")
    if per > 3:
        print("\n경고: 3초/건을 넘습니다. VRAM 포화나 CPU 폴백을 의심하세요.")
        print("  nvidia-smi로 다른 프로세스가 GPU를 쓰고 있지 않은지 확인하세요.")
else:
    print("측정할 문항이 없습니다 (이미 모두 끝났습니다)")

## 12. 본 추론

위 예상 시간이 납득되면 실행하세요. Colab은 유휴 상태로 두면 끊기므로 탭을 열어 두세요.
끊겼다면 런타임을 다시 잡고 **1번 설치 → 세션 재시작 → 2번부터 이 셀까지** 실행하면
Drive에 저장된 예측을 복구해 이어서 진행합니다.

In [ ]:
done_now = set()
with PRED_PATH.open(encoding="utf-8") as f:
    done_now = {json.loads(line)["id"] for line in f if line.strip()}
rest = [r for r in target_rows if r["id"] not in done_now]
print(f"남은 {len(rest)}개")

times = run_items(rest, "E1 제출 추론")
print("완료")

## 13. 제출 파일 생성

In [ ]:
pred_df = pd.read_json(PRED_PATH, lines=True).drop_duplicates("id", keep="last")
print(f"기록된 예측 {len(pred_df)}개 / 대상 {len(target_rows)}개")

if SUBMISSION_LIMIT:
    print("SUBMISSION_LIMIT이 설정되어 제출 파일을 만들지 않습니다.")
else:
    assert len(pred_df) >= len(target_df), \
        f"예측 부족 {len(pred_df)}/{len(target_df)}. 본 추론 셀을 다시 실행하세요"
    sub = target_df[["id"]].merge(pred_df[["id", "pred"]], on="id",
                                  how="left", validate="one_to_one")
    assert sub.pred.notna().all(), "비어 있는 예측이 있습니다"
    assert sub.pred.isin(ce.CHOICES).all(), \
        f"a~d 이외의 값: {sorted(set(sub.pred) - set(ce.CHOICES))}"
    if sample_df is not None:
        assert list(sub.id) == list(sample_df.id), "id 순서가 sample_submission과 다릅니다"

    if IS_TEST_SPLIT:
        name = submission_filename()
        out_path = OUT_DIR / name
        sub.rename(columns={"pred": "answer"}).to_csv(out_path, index=False, encoding="utf-8")
        shutil.copy2(out_path, DRIVE_DIR / name)
    else:
        out_path = None
        print("test.csv가 아니므로 제출 파일은 만들지 않습니다 (분포 점검·채점만 수행)")

    counts = sub.pred.value_counts().reindex(ce.CHOICES).fillna(0).astype(int)
    share = counts / counts.sum()
    print("\n예측 분포")
    print(pd.DataFrame({"건수": counts, "비중": share.round(3)}).to_string())
    if share.max() > 0.35:
        print(f"\n경고: '{share.idxmax()}'가 {share.max():.1%}로 쏠렸습니다. "
              "train 정답 분포의 최대 비중은 25.6%입니다. 파서와 폴백을 점검하세요")
    else:
        print(f"\n분포 정상 (최대 {share.max():.1%})")
    if out_path is not None:
        print("제출 파일:", out_path)
        print("Drive 사본:", DRIVE_DIR / out_path.name)

## 14. 선택 — 정답이 있는 CSV였을 때 채점

In [ ]:
if GOLD is None:
    print("정답이 없는 CSV입니다. 채점을 건너뜁니다.")
else:
    scored = pred_df[["id", "pred", "seconds"]].copy()
    scored["answer"] = scored.id.map(GOLD)
    scored = scored.dropna(subset=["answer"])
    scored["correct"] = scored.pred == scored.answer

    ce.save_analysis(scored, f"E1_colab_{STEM}", OUT_DIR, extra=dict(
        vlm_model=str(VLM_MODEL_DIR), vlm_4bit=VLM_4BIT, gpu=GPU_NAME,
        image_max_edge=ce.IMAGE_MAX_EDGE, answer_max_new_tokens=MAX_NEW_TOKENS,
        enable_thinking=ENABLE_THINKING, cot=False, source_csv=SUBMISSION_CSV.name,
        logit_fallbacks=int(pred_df.fallback.sum()), uses_image=True, uses_ocr=False))

    from math import sqrt
    k, n, z = int(scored.correct.sum()), len(scored), 1.96
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    print(f"\n정확도 {p:.4f} ({k}/{n}) / 95% CI [{centre - half:.4f}, {centre + half:.4f}] "
          f"= +-{half * 100:.2f}pp")
    print(f"로컬 검증 672문항(0.9420)과 비교: {p - 0.9420:+.4f}")

    for f in OUT_DIR.glob("*"):
        if f.is_file():
            shutil.copy2(f, DRIVE_DIR / f.name)
    print("Drive로 결과 복사 완료:", DRIVE_DIR)

## 15. 정리

메모리를 비웁니다. 다시 돌리려면 9번 셀부터 실행하세요.

In [ ]:
del model, proc
gc.collect(); torch.cuda.empty_cache()
print("해제 완료 / VRAM", f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

---

## 로컬 노트북과 다른 점

| | 로컬 | Colab |
| --- | --- | --- |
| GPU | RTX 5060 Ti 16 GB | A100 40 GB 권장 |
| 적재 | 4bit NF4 고정 | VRAM 감지 후 자동 (A100이면 bf16) |
| 최적화 커널 | 없음 (Windows) | `flash-linear-attention` 설치 |
| 모델 | `downloads/models/` 오프라인 | HF Hub에서 매 세션 다운로드 |
| 데이터 | `data/` | Drive zip 압축 해제 |
| 공통 코드 | `src/` 직접 | 저장소 clone 후 import |
| 출력 | `output/exp_E1_<대상>/` | `/content/out` + Drive 사본 |
| 파일명 | `..._zeroshot_v1.csv` | `..._zeroshot-bf16_v1.csv` (적재 방식 포함) |

프롬프트·이미지 규칙·파서·토큰 한도는 **동일**합니다. 그래야 로컬 검증 94.20%와 비교가 성립합니다.

## 주의

- **양자화가 다르면 결과도 조금 달라집니다.** A100 bf16 결과와 로컬 4bit 결과를 나란히 놓을 때는
  모델 크기 차이가 아니라 적재 방식 차이가 섞여 있다는 점을 명시하세요.
  파일명의 `-bf16` / `-4bit` 태그가 그 구분용입니다
- Colab은 유휴 시 끊깁니다. 끊기면 Drive 사본에서 복구해 이어서 진행합니다
- A100은 컴퓨팅 유닛 소모가 큽니다. 먼저 `SUBMISSION_LIMIT = 8`로 배관을 확인하세요
- 저장소가 비공개라 Colab에서 `git clone`이 인증 없이 실패합니다. `src/common_eval.py`를 Drive의 `SSAFY/AI_Challenge2/src/`에 올려 두는 쪽이 가장 간단합니다